In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import warnings

import arviz_plots as azp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.plotting import autocorrelation_plot
from sklearn.model_selection import StratifiedKFold

from compressor_fouling_modeling.utility import (
    calculate_residuals,
    plot_fouling_summary,
    predict_fouling_onset,
    train_model,
    visualize_data_folds,
    visualize_learning_curve,
    visualize_partial_dependency,
)

azp.style.use("dark_background")  # pick style of interest
%config InlineBackend.figure_format = 'retina'  # high resolution figures
warnings.filterwarnings("ignore")

In [ ]:
RANDOM_SEED = 14
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
# Define project root relative to notebook location
PROJECT_ROOT = Path().resolve().parents[0]  # goes up one level from /notebooks/
IMAGE_DIR = Path(PROJECT_ROOT / "results" / "plots")
DATA_DIR = PROJECT_ROOT / "data"

In [ ]:
# Check if the directory exists and create it if it doesn't
if not IMAGE_DIR.exists():
    try:
        IMAGE_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Directory '{IMAGE_DIR}' created successfully.")
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print(f"Directory '{IMAGE_DIR}' already exists.")

In [ ]:
X_baseline = pd.read_csv(DATA_DIR / "processed" / "X_baseline.csv", index_col=0, parse_dates=True).squeeze("columns")
y_baseline = pd.read_csv(DATA_DIR / "processed" / "y_baseline.csv", index_col=0, parse_dates=True).squeeze("columns")
X = pd.read_csv(DATA_DIR / "processed" / "X_full.csv", index_col=0, parse_dates=True).squeeze("columns")
y = pd.read_csv(DATA_DIR / "processed" / "y_full.csv", index_col=0, parse_dates=True).squeeze("columns")
baseline_mask = pd.read_csv(DATA_DIR / "processed" / "baseline_mask.csv", index_col=0, parse_dates=True).squeeze("columns")
shutin_mask = pd.read_csv(DATA_DIR / "processed" / "shutin_mask.csv", index_col=0, parse_dates=True).squeeze("columns")

# Linear Regression

Shuffle aggressively.

- It breaks "Time" so you only learn "Physics."
- It ensures you aren't just memorizing the sequence of events.

In [ ]:
n_splits = 3
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
cv_splits_stratified = []
stratify_on = X_baseline["Outlet_Pressure_SP"]
for train_idx, test_idx in skf.split(X_baseline, stratify_on):
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    cv_splits_stratified.append((train_idx.copy(), test_idx.copy()))

visualize_data_folds(X_baseline, y_baseline, cv_splits_stratified)

In [ ]:
lr_model = train_model(
    X_baseline, y_baseline, cv_splits_stratified, "lr", n_trials=1000
)

In [ ]:
visualize_learning_curve(
    lr_model,
    X_baseline,
    y_baseline,
    np.linspace(0.1, 1.0, 10),
    cv_splits_stratified,
    "neg_mean_absolute_error",
    True,
    RANDOM_SEED
)

Data Sufficiency: 

The lines are flat after ~60 samples (2 months of history). This answers the question: "Do I need more data?" Answer: No. Adding 1,000 more samples won't lower the error. We have fully extracted the signal. The remaining error (~2.5 psi) is just the irreducible noise of the sensors and the process.

In [ ]:
X_baseline

In [ ]:
visualize_partial_dependency(lr_model, X_baseline, RANDOM_SEED, "lr")

1. The Mystery of the "Four Parallel Bands"

    We will notice that in every plot, the blue lines are not scattered randomly. They are grouped into 4 distinct "bands" or clusters stacked vertically. What are these? These are our Operating Regimes (100 psi, 180 psi, 140 psi and 80 psi).

    The Insight: This visual proves that the model separates the "Regime" (the intercept/vertical level) from the "Physics" (the slope).
    Why it’s good: Since the lines within the bands are roughly parallel, it means the physical response is consistent.

    Example (Inlet Temperature): Whether the machine is running at 100 psi (bottom band) or 80 psi (top band), increasing the inlet temperature always increases the outlet pressure by the same relative amount (the slope is identical). This confirms our assumption that the physics is "Time-Invariant" and consistent across regimes.

2. Physics Validation (The Slopes)
    The Suction Boost (Inlet_Pressure plot):

    Slope: Positive.
    Physics: Validated. Higher inlet pressure gives the compressor a "head start," resulting in higher outlet pressure for the same effort.

3. The "Flat" Variables (Feature Selection Confirmation)
Look at Outlet_Flow_Rate and Outlet_Temperature.

Observation: They are perfectly flat.
Interpretation: The model has zeroed out these coefficients (or given them zero importance).
Why this is correct:
The model prefers Outlet_Flow_Rate_SP over Outlet_Flow_Rate. This is good! For a proactive "Digital Twin," you want to predict based on Control Intent (SP), not just sensor correlations. If the flow sensor fails, your model won't break because it relies on the Setpoint.

Final Verdict
We have successfully navigated the "Controller Trap."

We included Setpoints, which split the data into the correct vertical "bands."
We used Ridge/Splines, which captured the linear physics (slopes) within those bands.
We used ICE Plots, which prove that the Flow vs. Pressure relationship (the slope) holds true regardless of which Setpoint band you are in.

Baseline: The model predicts the "Blue Lines" (Healthy State).
Anomaly: When the machine fouls, the actual data points will start falling below the lowest blue band in the ICE plot (or drifting down across bands).
Action: Calculate Residual = Actual - Prediction. When the residual trend hits -3 or -4 sigma (based on your training noise), trigger the alert.

In [ ]:
lr_residuals_baseline = calculate_residuals(lr_model, X_baseline, y_baseline)
fig, ax = plt.subplots(1, 1)
autocorrelation_plot(lr_residuals_baseline, ax)
plt.title("Residual ACF")
plt.show()
plt.close(fig)
del fig, ax

The ACF plot (bottom image) shows that the autocorrelation is mostly contained within the confidence intervals (the gray dashed lines). This confirms that the residuals are essentially "white noise" (independent and identically distributed). This is the ideal prerequisite for CUSUM, which relies on the assumption of independent errors.

In [ ]:
lr_residuals = calculate_residuals(lr_model, X, y)


lr_model_fouling_dates, lr_model_cusum, lr_model_alarm_mask = predict_fouling_onset(
    lr_residuals, baseline_mask, shutin_mask, "neg"
)

fname = Path(IMAGE_DIR / "fouling_summary.png")
plot_fouling_summary(
    y,
    y - lr_residuals,
    baseline_mask,
    shutin_mask,
    lr_model_cusum,
    lr_model_alarm_mask,
    save=True,
    fname=fname
)

In [ ]:
spline_model = train_model(
    X_baseline, y_baseline, cv_splits_stratified, "spline", n_trials=1000
)

In [ ]:
visualize_learning_curve(
    spline_model,
    X_baseline,
    y_baseline,
    np.linspace(0.1, 1.0, 10),
    cv_splits_stratified,
    "neg_mean_absolute_error",
    True,
    RANDOM_SEED,
)

In [ ]:
visualize_partial_dependency(spline_model, X_baseline, RANDOM_SEED, "spline")

As for each data shuffle, I get different knot configuration (but degree=1 is almost constant), data seems strictly linear and using spline for such limited data results in unstability. I continue with linear model in production.

### Assumptions

Here are the key assumptions you are making by adopting this "Global Shuffled Baseline" strategy.

It is crucial to document these, as violating any of them in the future could lead to false alarms or missed detections.
1. Physics is Time-Invariant (The "Memoryless" Assumption)

    This is the justification for shuffling your time-series data.

- The Assumption: You assume that the physics of a healthy compressor does not change over time.
- The Implication: A datapoint collected in August (State A) is physically identical to a datapoint collected in December (State A), provided the machine was healthy in both instances. Therefore, the chronological order of training samples does not matter for learning the input/output relationship—only the state matters.
2. Completeness of Operating Regimes (Interpolation vs. Extrapolation)
- The Assumption: You assume that the "Healthy" training period (Aug–Dec) contains all the Setpoints (SP) and Flow regimes that the machine will encounter in the future.
- The Implication: Your model has learned to predict behavior at SP=80, 100, 140 and 180.
If the operators decide to run the machine at SP=110 (between known points), the model will work (Interpolation).
If they decide to run at SP=200 (outside known points), the model will likely fail or give high errors because it is forced to extrapolate into unknown territory.
3. Input Sensor Integrity (Trust in Upstream Data)
- The Assumption: You assume that while the Compressor might foul, the Sensors (Inlet Flow, Inlet Pressure, Temperature) remain perfectly healthy and calibrated.
- The Implication: The model relies on the Inlet Pressure to predict the Outlet Pressure. If the Inlet Pressure sensor starts drifting due to its own electrical fault, your model will produce a wrong prediction, leading to a False Positive alert for compressor fouling.
4. Stationarity of Control Logic
- The Assumption: You assume the PID controller tuning (the logic that drives the machine to meet the Setpoint) has not been changed between the Training period and the Test period.
- The Implication: If a control engineer re-tuned the PID loop in January to make the compressor react faster or slower, the relationship between Setpoint and Actual_Pressure changes. Your model would interpret this new control behavior as an anomaly.
5. Thermodynamic Equilibrium (Steady State)
- The Assumption: You assume the daily samples represent "Steady State" operation, not transient startups or shutdowns.
- The Implication: Since you are using daily batch data (or filtered data), you assume you aren't capturing the exact moment the machine is ramping up from 0 to 100. If a data point captures a transient ramp-up, the pressure will naturally be lagging the setpoint. The model might flag this lag as a performance deficit (fouling) unless those transient points are rigorously filtered out.
6. Linearity of Correction Factors
- The Assumption: By using Ridge Regression (or similar linear models), you assume that the impact of secondary variables (Ambient Humidity, Off-Gas Fraction) is roughly linear or additive.
- The Implication: While fluid dynamics are non-linear, for a proactive maintenance model, we assume that within the standard operating window, the relationship is "linear enough" to set a baseline. We assume we don't need a complex Neural Network to capture extreme edge-case thermodynamics.

Summary for Stakeholders
If you need to present this, you can summarize it as:

"We are assuming the machine's behavior is consistent when healthy, regardless of the date. We are also assuming that we have seen examples of all standard operating speeds. If the machine is pushed to a speed we haven't seen before, or if the inlet sensors fail, the model will require retraining."